In [ ]:
from config import client, WORKDIR, MODEL, MODES, SKILLS_DIR, TASKS_DIR
from better_output import show_whole
from TODO import TodoManager
from Memory import MemoryManager, MEMORY_GUIDANCE
from Skill import SkillDocument, SkillMainfest, SkillRegistry
from Subagent import AgentTemplate
from Tool import run_bash, run_edit, run_read, run_write, PermissionManager, run_save_memory
from Hook import HookManager
from Task import TaskManager
from Backgroud_task import BackgroundManager
from CronScheduler import CronScheduler
from Error_recovery import backoff_delay, MAX_RECOVERY_ATTEMPTS, BACKOFF_BASE_DELAY, BACKOFF_MAX_DELAY, TOKEN_THRESHOLD, CONTINUATION_MESSAGE
from System_prompt import build_system_reminder, SystemPromptBuilder, DYNAMIC_BOUNDARY
from Compact import (CompactState, estimate_context_size, micro_compact, 
                    compact_history, CONTEXT_LIMIT)

from anthropic import APIError
import time

SUBAGENT_SYSTEM = f"You are a coding subagent at {WORKDIR}. Complete the given task, then summarize your findings."

SKILL_REGISTRY = SkillRegistry(SKILLS_DIR)

TODO = TodoManager()

TASKS = TaskManager(TASKS_DIR)

BG = BackgroundManager()

scheduler = CronScheduler()

TOOL_HANDLERS = {
    "bash":         lambda **kw: run_bash(kw["command"]),
    "read_file":    lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file":   lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file":    lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
    "todo":         lambda **kw: TODO.update(kw["item"]),
    "load_skill":   lambda **kw: SKILL_REGISTRY.load_full_text(kw["name"]),
    "save_memory":  lambda **kw: run_save_memory(kw["name"], kw["description"], kw["type"], kw["content"]),
    "task_create": lambda **kw: TASKS.create(kw["subject"], kw.get("description", "")),
    "task_update": lambda **kw: TASKS.update(kw["task_id"], kw.get("status"), kw.get("owner"), kw.get("addBlockedBy"), kw.get("addBlocks")),
    "task_list":   lambda **kw: TASKS.list_all(),
    "task_get":    lambda **kw: TASKS.get(kw["task_id"]),
    "background_run":   lambda **kw: BG.run(kw["command"]),
    "check_background": lambda **kw: BG.check(kw.get("task_id")),
    "cron_create": lambda **kw: scheduler.create(
        kw["cron"], kw["prompt"], kw.get("recurring", True), kw.get("durable", False)),
    "cron_delete": lambda **kw: scheduler.delete(kw["id"]),
    "cron_list":   lambda **kw: scheduler.list_tasks(),
}
    
TOOLS = [
    {"name": "bash","description": "Run a shell command.",
    "input_schema": {"type": "object","properties": {"command": {"type": "string"}},
    "required": ["command"],},},
    
    {"name": "read_file","description": "Read file contents.",
    "input_schema": {"type": "object","properties": {"path": {"type": "string"},"limit": {"type": "integer"},},
    "required": ["path"],},},
    
    {"name": "write_file","description": "Write content to a file.",
    "input_schema": {"type": "object","properties": {"path": {"type": "string"},"content": {"type": "string"},},
    "required": ["path", "content"],},},
    
    {"name": "edit_file","description": "Replace exact text in a file once.",
     "input_schema": {"type": "object","properties": {"path": {"type": "string"},"old_text": {"type": "string"},"new_text": {"type": "string"},},
    "required": ["path", "old_text", "new_text"],},},
    
    {
        "name": "todo",
        "description": "Rewrite the current session plan for multi-step work.",
        "input_schema": {
            "type": "object",
            "properties": {
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "content": {"type": "string"},
                            "status": {
                                "type": "string",
                                "enum": ["pending", "in_progress", "completed"],
                            },
                            "activeForm": {
                                "type": "string",
                                "description": "Optional present-continuous label.",
                            },
                        },
                        "required": ["content", "status"],
                    },
                },
            },
            "required": ["items"],
        },
    },
    
    {"name": "load_skill","description": "Load the full body of a named skill into the current context.",
    "input_schema": {"type": "object","properties": {"name": {"type": "string"}},
    "required": ["name"],},},

    {"name": "compact", "description": "Summarize earlier conversation so work can continue in a smaller context.",
    "input_schema": {"type": "object", "properties": {"focus": {"type": "string"},},},},

    {"name": "save_memory", "description": "Save a persistent memory that survives across sessions.",
     "input_schema": {"type": "object", "properties": {
         "name": {"type": "string", "description": "Short identifier (e.g. prefer_tabs, db_schema)"},
         "description": {"type": "string", "description": "One-line summary of what this memory captures"},
         "type": {"type": "string", "enum": ["user", "feedback", "project", "reference"],
                  "description": "user=preferences, feedback=corrections, project=non-obvious project conventions or decision reasons, reference=external resource pointers"},
         "content": {"type": "string", "description": "Full memory content (multi-line OK)"},
     }, "required": ["name", "description", "type", "content"]}},

     {"name": "task_create", "description": "Create a new task.",
     "input_schema": {"type": "object", "properties": {"subject": {"type": "string"}, "description": {"type": "string"}}, "required": ["subject"]}},
    {"name": "task_update", "description": "Update a task's status, owner, or dependencies.",
     "input_schema": {"type": "object", "properties": {"task_id": {"type": "integer"}, "status": {"type": "string", "enum": ["pending", "in_progress", "completed", "deleted"]}, "owner": {"type": "string", "description": "Set when a teammate claims the task"}, "addBlockedBy": {"type": "array", "items": {"type": "integer"}}, "addBlocks": {"type": "array", "items": {"type": "integer"}}}, "required": ["task_id"]}},
    {"name": "task_list", "description": "List all tasks with status summary.",
     "input_schema": {"type": "object", "properties": {}}},
    {"name": "task_get", "description": "Get full details of a task by ID.",
     "input_schema": {"type": "object", "properties": {"task_id": {"type": "integer"}}, "required": ["task_id"]}},

     {"name": "background_run", "description": "Run command in background thread. Returns task_id immediately.",
     "input_schema": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}},
    {"name": "check_background", "description": "Check background task status. Omit task_id to list all.",
     "input_schema": {"type": "object", "properties": {"task_id": {"type": "string"}}}},

    {"name": "cron_create", "description": "Schedule a recurring or one-shot task with a cron expression.",
     "input_schema": {"type": "object", "properties": {
         "cron": {"type": "string", "description": "5-field cron expression: 'min hour dom month dow'"},
         "prompt": {"type": "string", "description": "The prompt to inject when the task fires"},
         "recurring": {"type": "boolean", "description": "true=repeat, false=fire once then delete. Default true."},
         "durable": {"type": "boolean", "description": "true=persist to disk, false=session-only. Default false."},
     }, "required": ["cron", "prompt"]}},
    {"name": "cron_delete", "description": "Delete a scheduled task by ID.",
     "input_schema": {"type": "object", "properties": {
         "id": {"type": "string", "description": "Task ID to delete"},
     }, "required": ["id"]}},
    {"name": "cron_list", "description": "List all scheduled tasks.",
     "input_schema": {"type": "object", "properties": {}}},
]


CHILD_TOOLS = [
    {"name": "bash", "description": "Run a shell command.",
     "input_schema": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}},
    {"name": "read_file", "description": "Read file contents.",
     "input_schema": {"type": "object", "properties": {"path": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["path"]}},
    {"name": "write_file", "description": "Write content to file.",
     "input_schema": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]}},
    {"name": "edit_file", "description": "Replace exact text in file.",
     "input_schema": {"type": "object", "properties": {"path": {"type": "string"}, "old_text": {"type": "string"}, "new_text": {"type": "string"}}, "required": ["path", "old_text", "new_text"]}},
]

PARENT_TOOLS = CHILD_TOOLS + [
    {"name": "task", "description": "Spawn a subagent with fresh context. It shares the filesystem but not conversation history.",
     "input_schema": {"type": "object", "properties": {"prompt": {"type": "string"}, "description": {"type": "string", "description": "Short description of the task"}}, "required": ["prompt"]}},
]

prompt_builder = SystemPromptBuilder(workdir=WORKDIR, tools = TOOLS)

SYSTEM = prompt_builder.build()

def extract_text(content) -> str:
    if not isinstance(content, list):
        return ""
    texts = []
    for block in content:
        text = getattr(block, "text", None)
        if text:
            texts.append(text)
    return "\n".join(texts).strip()

def run_subagent(prompt: str) -> str:
    sub_messages = [{"role": "user", "content": prompt}]
    for _ in range(30):
        response = client.messages.create(
            model = "deepseek-chat", system=SUBAGENT_SYSTEM, messages=sub_messages,
            tools=CHILD_TOOLS, max_tokens=8000,
        )
        sub_messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason != "tool_use":
            break
        results = []
        for block in response.content:
            if block.type == "tool_use":
                handler = TOOL_HANDLERS.get(block.name)
                ouput = handler(**block.input) if handler else f"Unknown tool: {block.name}"
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(ouput)[:50000]})
        sub_messages.append({"role": "user", "content": results})

        show_whole(sub_messages, "sub_messages")
    return "".join(b.text for b in response.content if hasattr(b, "text")) or "(no summary)"

def agent_loop(messages: list, state: CompactState, perms: PermissionManager, hooks: HookManager) -> None:
    
    max_output_recovery_count = 0

    while True:
        response = None

        messages[:] = micro_compact(messages)

        if estimate_context_size(messages) > CONTEXT_LIMIT:
            print("[auto compect]")
            messages[:] = compact_history(messages, state)
        
        notifs = BG.drain_notifications()

        if notifs:
            notif_text = "\n".join(
                f"[bg:{n['task_id']}] {n['status']}: {n['preview']} "
                f"(output_file={n['output_file']})"
                for n in notifs
            )
            messages.append({"role": "user", "content": f"<background-results>\n{notif_text}\n</background-results>"})
        
        notifications = scheduler.drain_notifications()
        
        for note in notifications:
            print(f"[Cron notification] {note[:100]}")
            messages.append({"role": "user", "content": note})

        for attempt in range(MAX_RECOVERY_ATTEMPTS + 1):
            try:
                response = client.messages.create(
                    model=MODEL,
                    system=SYSTEM,
                    messages=messages,
                    tools=TOOLS,
                    max_tokens=8000,
                )
                break
            
            except APIError as e:
                error_body = str(e).lower()

                if "overlong_prompt" in error_body or ("prompt" in error_body and "long" in error_body):
                    print(f"[Recovery] Prompt too long. Compacting... (attempt {attempt + 1})")
                    messages[:] = compact_history(messages)
                    continue

                # Strategy 3: connection/rate errors -> backoff
                if attempt < MAX_RECOVERY_ATTEMPTS:
                    delay = backoff_delay(attempt)
                    print(f"[Recovery] API error: {e}. "
                          f"Retrying in {delay:.1f}s (attempt {attempt + 1}/{MAX_RECOVERY_ATTEMPTS})")
                    time.sleep(delay)
                    continue

                # All retries exhausted
                print(f"[Error] API call failed after {MAX_RECOVERY_ATTEMPTS} retries: {e}")
                return
            
            except (ConnectionError, TimeoutError, OSError) as e:
                # Strategy 3: network-level errors -> backoff
                if attempt < MAX_RECOVERY_ATTEMPTS:
                    delay = backoff_delay(attempt)
                    print(f"[Recovery] Connection error: {e}. "
                          f"Retrying in {delay:.1f}s (attempt {attempt + 1}/{MAX_RECOVERY_ATTEMPTS})")
                    time.sleep(delay)
                    continue

                print(f"[Error] Connection failed after {MAX_RECOVERY_ATTEMPTS} retries: {e}")
                return
            
        if response is None:
            print("[Error] No response received.")
            return
        
        messages.append({"role": "assistant", "content": response.content})

        # -- Strategy 1: max_tokens recovery --
        # 解决输出太长的问题，因为max_token=8000
        # 设置最大重复次数，防止模型一直说
        if response.stop_reason == "max_tokens":
            max_output_recovery_count += 1
            if max_output_recovery_count <= MAX_RECOVERY_ATTEMPTS:
                print(f"[Recovery] max_tokens hit "
                      f"({max_output_recovery_count}/{MAX_RECOVERY_ATTEMPTS}). "
                      "Injecting continuation...")
                messages.append({"role": "user", "content": CONTINUATION_MESSAGE})
                continue  # retry the loop
            else:
                print(f"[Error] max_tokens recovery exhausted "
                      f"({MAX_RECOVERY_ATTEMPTS} attempts). Stopping.")
                return
            
        max_output_recovery_count = 0

        if response.stop_reason != "tool_use":
            return
        
        results = []

        manual_compact = False
        compact_focus = None

        for block in response.content:
            if block.type != "tool_use":
                continue

            tool_input = dict(block.input or {})
            ctx = {"tool_name": block.name, "tool_input": tool_input}

            pre_result = hooks.run_hooks("PreToolUse", ctx)

            for msg in pre_result.get("message", []):
                results.append({
                    "type": "tool_result", "tool_use_id": block.id,
                    "content": f"[Hook message]: {msg}"
                })
            
            if pre_result.get("blocked"):
                reason = pre_result.get("block_reason", "Block by hook")
                output = f"Tool blocked by PreToolUse hook: {reason}"
                results.append({
                    "type": "tool_result", "tool_use_id": block.id,
                    "content": output,
                })
                continue

            decision = perms.check(block.name, block.input or [])

            if decision["behavior"] == "deny":
                output = f"Permission denied: {decision['reason']}"
                print(f"  [DENIED] {block.name}: {decision['reason']}")

            elif decision["behavior"] == "ask":
                if perms.ask_user(block.name, block.input or {}):
                    handler = TOOL_HANDLERS.get(block.name)
                    output = handler(**(tool_input or {})) if handler else f"Unknown: {block.name}"
                    print(f"> {block.name}: {str(output)[:200]}")
                else:
                    output = f"Permission denied by user for {block.name}"
                    print(f"  [USER DENIED] {block.name}")
            else:
                handler = TOOL_HANDLERS.get(block.name)
                try:
                    output = handler(**tool_input) if handler else f"Unknown tool: {block.name}"
                except Exception as exc:
                    output = f"Error: {exc}"

                print(f"> {block.name}: {str(output)[:200]}")
            
            ctx["tool_output"] = output
            post_result = hooks.run_hooks("PostToolUse", ctx)

            for msg in post_result.get("message", []):
                output += f"\n[Hook note]: {msg}"
                       
            if block.name == "compact":
                manual_compact = True
                compact_focus = (block.input or {}).get("focus")

            results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(output),
            })
            
        messages.append({"role": "user", "content": results})

        if manual_compact:
            print("[manual compact]")
            messages[:] = compact_history(messages, state, focus=compact_focus)

        show_whole(messages, "messages")

if __name__ == "__main__":

    print("Permission modes: default, plan, auto")
    mode_input = input("Mode (default): ").strip().lower() or "default"
    if mode_input not in MODES:
        mode_input = "default"
    perms = PermissionManager(mode=mode_input)
    print(f"[Permission mode: {mode_input}]")

    full_prompt = prompt_builder.build()
    section_count = full_prompt.count("\n# ")
    print(f"[System prompt assembled: {len(full_prompt)} chars, ~{section_count} sections]")

    history = []
    compact_state = CompactState()
    hooks = HookManager()
    while True:
        try:
            query = input("\033[36ms05 >> \033[0m")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() in ("q", "exit", ""):
            break

        if query.startswith("/mode"):
            parts = query.split()
            if len(parts) == 2 and parts[1] in MODES:
                perms.mode = parts[1]
                print(f"[Switched to {parts[1]} mode]")
            else:
                print(f"Usage: /mode <{'|'.join(MODES)}>")
            continue

        # /rules command to show current rules
        if query.strip() == "/rules":
            for i, rule in enumerate(perms.rules):
                print(f"  {i}: {rule}")
            continue

        if query.strip() == "/sections":
            prompt = prompt_builder.build()
            for line in prompt.splitlines():
                if line.startswith("# ") or line == DYNAMIC_BOUNDARY:
                    print(f"  {line}")
            continue
        
        history.append({"role": "user", "content": query})
        agent_loop(history, compact_state, perms, hooks)
        final_text = extract_text(history[-1]["content"])
        if final_text:
            print(final_text)
        
        print()

Permission modes: default, plan, auto
[Permission mode: default]
[System prompt assembled: 1437 chars, ~2 sections]
你好！有什么我可以帮你的吗？😊


 [Permission] todo: {"items": [{"content": "了解用户需求，制定工作计划", "status": "in_progress"}]}
  [USER DENIED] todo


好的，请问你有什么任务或项目需要我帮忙？可以告诉我具体的目标，我来帮你制定计划并一步步执行。

Usage: /mode <default|plan|auto>
[Switched to auto mode]
  # Available tools
  === DYNAMIC_BOUNDARY ===
  # Dynamic context
